# 06 - Binary NLI Fine-Tuning (ContraDoc)

Fine-tune `cross-encoder/nli-deberta-v3-base` for binary contradiction detection on
the leakage-controlled splits prepared in `nli_finetune_data.ipynb`.

The pretrained checkpoint has 3 NLI classes (contradiction / entailment / neutral). We
replace the classification head with a 2-class head (`0 = not_contradiction`, `1 = contradiction`)
and let `ignore_mismatched_sizes=True` reinitialize it. Encoder weights transfer; the head re-trains.

**Inputs (held-out, base_doc_id-disjoint):**
- `data/processed/ContraDoc/nli/nli_train.csv`
- `data/processed/ContraDoc/nli/nli_val.csv`
- `data/processed/ContraDoc/nli/nli_test.csv`

**Output:** `../fine-tuning/models/nli_binary/` (gitignored)


In [1]:
import csv
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import classification_report
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

NLI_DIR = Path("data/processed/ContraDoc/nli")
TRAIN_CSV = NLI_DIR / "nli_train.csv"
VAL_CSV   = NLI_DIR / "nli_val.csv"
TEST_CSV  = NLI_DIR / "nli_test.csv"

MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
OUTPUT_DIR = Path("../fine-tuning/models/nli_binary")
SEED = 42
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if device.type == "cuda":
    print(f"gpu:    {torch.cuda.get_device_name(0)}")

device: cuda
gpu:    NVIDIA GeForce RTX 4060 Laptop GPU


## Load splits

Splits were built in `nli_finetune_data.ipynb` with strict `base_doc_id` disjointness
between train / val / test. Re-assert that here so any future regeneration is caught.

In [2]:
def load_pairs(csv_path):
    rows = []
    with csv_path.open(encoding="utf-8") as f:
        for r in csv.DictReader(f):
            rows.append({
                "premise": r["premise"],
                "hypothesis": r["hypothesis"],
                "label": int(r["label"]),
                "doc_id": r["doc_id"],
                "base_doc_id": r["base_doc_id"],
                "contra_type": r["contra_type"],
                "kind": r["kind"],
            })
    return rows

train_pairs = load_pairs(TRAIN_CSV)
val_pairs   = load_pairs(VAL_CSV)
test_pairs  = load_pairs(TEST_CSV)

def stats(name, pairs):
    pos = sum(p["label"] for p in pairs)
    print(f"{name:6s}: {len(pairs):4d} pairs  (pos={pos}, neg={len(pairs)-pos})")

stats("train", train_pairs)
stats("val",   val_pairs)
stats("test",  test_pairs)

# Leakage: in-domain doc_id-disjoint policy.
# Strict: pairwise doc_id-disjoint between splits + train/val base-disjoint.
# Soft (reported, not asserted): train/test bases CAN overlap (intentional in-domain training).
train_doc_ids = {p["doc_id"] for p in train_pairs}
val_doc_ids   = {p["doc_id"] for p in val_pairs}
test_doc_ids  = {p["doc_id"] for p in test_pairs}
assert not (train_doc_ids & val_doc_ids),  f"train/val doc_id overlap: {train_doc_ids & val_doc_ids}"
assert not (train_doc_ids & test_doc_ids), f"train/test doc_id overlap: {train_doc_ids & test_doc_ids}"
assert not (val_doc_ids & test_doc_ids),   f"val/test doc_id overlap: {val_doc_ids & test_doc_ids}"

train_bases = {p["base_doc_id"] for p in train_pairs}
val_bases   = {p["base_doc_id"] for p in val_pairs}
test_bases  = {p["base_doc_id"] for p in test_pairs}
assert not (train_bases & val_bases), f"train/val base overlap: {train_bases & val_bases}"

shared_train_test_bases = train_bases & test_bases
shared_val_test_bases   = val_bases & test_bases
print()
print(f"doc_id disjoint: train={len(train_doc_ids)}, val={len(val_doc_ids)}, test={len(test_doc_ids)} (asserted)")
print(f"base disjoint train/val: {len(train_bases)} train bases, {len(val_bases)} val bases (asserted)")
print(f"in-domain overlap (allowed): train ∩ test = {len(shared_train_test_bases)} bases, val ∩ test = {len(shared_val_test_bases)} bases")

train :  534 pairs  (pos=90, neg=444)
val   :  127 pairs  (pos=23, neg=104)
test  :  667 pairs  (pos=103, neg=564)

doc_id disjoint: train=105, val=24, test=135 (asserted)
base disjoint train/val: 80 train bases, 15 val bases (asserted)
in-domain overlap (allowed): train ∩ test = 34 bases, val ∩ test = 8 bases


## Tokenizer + model

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)
model.config.id2label = {0: "not_contradiction", 1: "contradiction"}
model.config.label2id = {"not_contradiction": 0, "contradiction": 1}

assert model.classifier.weight.shape[0] == 2, "Head not binary"
print(f"params: {sum(p.numel() for p in model.parameters()):,}")
print(f"head:   {tuple(model.classifier.weight.shape)}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


params: 184,423,682
head:   (2, 768)


## Datasets

In [4]:
class PairDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len):
        enc = tokenizer(
            [p["premise"]    for p in pairs],
            [p["hypothesis"] for p in pairs],
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt",
        )
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.token_type_ids = enc.get("token_type_ids")
        self.labels         = torch.tensor([p["label"] for p in pairs], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {
            "input_ids":      self.input_ids[i],
            "attention_mask": self.attention_mask[i],
            "labels":         self.labels[i],
        }
        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[i]
        return item

train_ds = PairDataset(train_pairs, tokenizer, MAX_LEN)
val_ds   = PairDataset(val_pairs,   tokenizer, MAX_LEN)
test_ds  = PairDataset(test_pairs,  tokenizer, MAX_LEN)
print(f"train_ds={len(train_ds)}  val_ds={len(val_ds)}  test_ds={len(test_ds)}")

train_ds=534  val_ds=127  test_ds=667


## Train

In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.350781,0.276520
2,0.219939,0.195689
3,0.184330,0.184568


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=201, training_loss=0.26903138357905015, metrics={'train_runtime': 68.6218, 'train_samples_per_second': 23.345, 'train_steps_per_second': 2.929, 'total_flos': 210755734935552.0, 'train_loss': 0.26903138357905015, 'epoch': 3.0})

## Validation report

In [6]:
val_out = trainer.predict(val_ds)
y_pred = np.argmax(val_out.predictions, axis=1)
y_true = [p["label"] for p in val_pairs]
print(classification_report(
    y_true, y_pred,
    target_names=["not_contradiction", "contradiction"],
    digits=3,
))

                   precision    recall  f1-score   support

not_contradiction      0.962     0.981     0.971       104
    contradiction      0.905     0.826     0.864        23

         accuracy                          0.953       127
        macro avg      0.934     0.903     0.918       127
     weighted avg      0.952     0.953     0.952       127



## Test report (held-out)

This is the only place we touch the test split. The 150 balanced docs that produced
these test pairs have base_doc_ids that never appear in train or val.

In [7]:
test_out = trainer.predict(test_ds)
y_pred = np.argmax(test_out.predictions, axis=1)
y_true = [p["label"] for p in test_pairs]
print(classification_report(
    y_true, y_pred,
    target_names=["not_contradiction", "contradiction"],
    digits=3,
))

                   precision    recall  f1-score   support

not_contradiction      0.936     0.979     0.957       564
    contradiction      0.844     0.631     0.722       103

         accuracy                          0.925       667
        macro avg      0.890     0.805     0.839       667
     weighted avg      0.921     0.925     0.920       667



## Save model

In [8]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"saved to {OUTPUT_DIR.resolve()}")
print("files:", sorted(f.name for f in OUTPUT_DIR.iterdir()))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\fine-tuning\models\nli_binary
files: ['checkpoints', 'config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
